# 04. attention.py: как токены смотрят друг на друга

Сначала реализуем attention из трёх операций, затем сравним с SDPA. Объясним движки FlashAttention/Sage/SDPA и причину большого расхода памяти в видео.


### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


### 1. Query, Key, Value — вопрос, адрес, содержимое

Query каждого токена сравнивается с Key других. Softmax превращает оценки в веса; Value смешиваются с этими весами. Деление на sqrt(head_dim) удерживает величины оценок разумными.


In [ ]:
import torch.nn.functional as F
q=torch.randn(1,2,5,4); k=torch.randn(1,2,5,4); v=torch.randn(1,2,5,4)
scores=q@k.transpose(-1,-2)/np.sqrt(q.shape[-1])
weights=scores.softmax(-1); manual=weights@v
print('scores:',scores.shape,'sum rows:',weights.sum(-1))
plt.imshow(weights[0,0]); plt.xlabel('key'); plt.ylabel('query'); plt.show()


### 2. Воспроизводим SDPA

PyTorch SDPA вычисляет ту же операцию, но может выбирать эффективный backend. В upstream вход [B,L,H,D] транспонируют в [B,H,L,D]. Проверка ниже ловит ошибки осей и масштаба.


In [ ]:
sdpa=F.scaled_dot_product_attention(q,k,v)
print('max error:',(manual-sdpa).abs().max().item())
assert torch.allclose(manual,sdpa,atol=1e-5)
source_excerpt('kandinsky/models/attention.py','def sdpa',21)


### 3. Mask: запрещённые связи

True в bool SDPA mask разрешает связь. В float-маске 0 разрешает, −inf запрещает. Мы запрещаем последний ключ и сравниваем API с явным softmax. Нельзя подставлять bool-маску с противоположным смыслом из другого API.


In [ ]:
mask=torch.tensor([True,True,True,True,False])[None,None,None,:]
masked_manual=scores.masked_fill(~mask,float('-inf')).softmax(-1)@v
masked_sdpa=F.scaled_dot_product_attention(q,k,v,attn_mask=mask)
assert torch.allclose(masked_manual,masked_sdpa,atol=1e-5)
print('Изменение после mask:',(masked_sdpa-manual).abs().mean().item())


### 4. Query chunking без потери ключей

При dense attention каждый запрос независим от других запросов. Поэтому можно обрабатывать Query порциями, сохраняя полный K/V. Это снижает размер временной матрицы, но не число сравнений. Разделить K/V и независимо применить softmax было бы неверно.


In [ ]:
chunked=torch.cat([F.scaled_dot_product_attention(q[:,:,i:i+2],k,v) for i in range(0,5,2)],dim=2)
assert torch.allclose(chunked,sdpa,atol=1e-5)
print('chunked max error:',(chunked-sdpa).abs().max().item())


### 5. Масштаб видео

Увеличение H и W вдвое даёт в 4 раза больше токенов и до 16 раз больше элементов dense attention. FlashAttention избегает хранения полной матрицы, но dense вычисления остаются квадратичными. Наш Mac adapter использует query chunks; он может быть медленнее оптимизированного CUDA.


In [ ]:
for seconds,H,W in [(1,256,256),(5,512,512),(5,512,768),(10,512,768)]:
    n=(seconds*6+1)*(H//16)*(W//16)
    print((seconds,H,W),'tokens',n,'dense/head FP32 GiB',round(n*n*4/2**30,2))


### 6. SelfAttentionEngine: выбор реализации

flash_attention_2/3 и sage требуют соответствующих библиотек. auto проверяет их наличие и иначе выбирает sdpa. MPS-путь курса выбирает только SDPA без compile. В CUDA Colab начните с SDPA, а затем измеряйте альтернативы по одинаковым prompt/seed.


In [ ]:
source_excerpt('kandinsky/models/attention.py','class SelfAttentionEngine',56)


#


In [ ]:
#


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
